In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

In [0]:
# Sample Data: Order ID and an Array of Items
data = [
    ("O-1001", ["Laptop", "Mouse", "Keyboard"]),
    ("O-1002", ["Monitor"]),
    ("O-1003", ["Desk", "Chair"])
]

# Define Schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("items", ArrayType(StringType()), True)
])

df = spark.createDataFrame(data, schema)
df.show(truncate=False)

+--------+-------------------------+
|order_id|items                    |
+--------+-------------------------+
|O-1001  |[Laptop, Mouse, Keyboard]|
|O-1002  |[Monitor]                |
|O-1003  |[Desk, Chair]            |
+--------+-------------------------+



In [0]:
display(df)

order_id,items
O-1001,"List(Laptop, Mouse, Keyboard)"
O-1002,List(Monitor)
O-1003,"List(Desk, Chair)"


In [0]:
# The PySpark Solution
flattened_df = df.select("order_id", explode("items").alias("single_item"))

flattened_df.show()

+--------+-----------+
|order_id|single_item|
+--------+-----------+
|  O-1001|     Laptop|
|  O-1001|      Mouse|
|  O-1001|   Keyboard|
|  O-1002|    Monitor|
|  O-1003|       Desk|
|  O-1003|      Chair|
+--------+-----------+



In [0]:
display(flattened_df)

order_id,single_item
O-1001,Laptop
O-1001,Mouse
O-1001,Keyboard
O-1002,Monitor
O-1003,Desk
O-1003,Chair


In [0]:
data = [
    ("Alice", ["Math", "Science"]),
    ("Bob", ["History"]),
    ("Charlie", []) # Empty array
]
columns = ["Student", "Subjects"]

df = spark.createDataFrame(data, columns)

# 3. Apply the explode function
df_exploded = df.select("Student", explode("Subjects").alias("Subject"))

df_exploded.show()

+-------+-------+
|Student|Subject|
+-------+-------+
|  Alice|   Math|
|  Alice|Science|
|    Bob|History|
+-------+-------+



In [0]:
# This keeps Charlie in the dataset with a null Subject
df_safe = df.select("Student", explode_outer("Subjects").alias("Subject"))
df_safe.show()

+-------+-------+
|Student|Subject|
+-------+-------+
|  Alice|   Math|
|  Alice|Science|
|    Bob|History|
|Charlie|   NULL|
+-------+-------+



In [0]:
data_map = [
    ("Alice", {"Math": 95, "Science": 90}),
    ("Bob", {"History": 85})
]

df_map = spark.createDataFrame(data_map, ["Student", "Scores"])

# Explode generates 'key' and 'value' columns automatically
df_map_exploded = df_map.select("Student", explode("Scores"))

df_map_exploded.show()

+-------+-------+-----+
|Student|    key|value|
+-------+-------+-----+
|  Alice|   Math|   95|
|  Alice|Science|   90|
|    Bob|History|   85|
+-------+-------+-----+

